In [ ]:
import fastf1
import pandas as pd

TEMPORADA_INICIO = 2026
TEMPORADA_FIM    = 2026
RODADA_INICIO    = 6
RODADA_FIM       = None

def extrair_clima_fastf1(temp_inicio, temp_fim,rodada_inicio, rodada_fim):
    registros = []

    for temporada in range(temp_inicio, temp_fim + 1):
        schedule = fastf1.get_event_schedule(temporada, include_testing=False)

        for _, evento in schedule.iterrows():
            rodada = evento['RoundNumber']

            if rodada < rodada_inicio:
                continue
            if rodada_fim is not None and rodada > rodada_fim:
                continue

            try:
                sessao = fastf1.get_session(temporada, rodada, 'R')
                sessao.load(weather=True, laps=False, telemetry=False, messages=False)

                weather = sessao.weather_data

                if weather is None or weather.empty:
                    print(f"Sem dados de clima: {temporada} R{rodada}")
                    continue

                total_voltas   = len(weather)
                voltas_chuva   = weather['Rainfall'].sum()
                perc_chuva     = (voltas_chuva / total_voltas * 100) if total_voltas > 0 else 0.0

                registros.append({
                    'temporada'       : temporada,
                    'rodada'          : rodada,
                    'nome_gp'         : evento['EventName'],
                    'pais'            : evento['Country'],
                    'circuito'        : evento['Location'],
                    'data_corrida'    : evento['Session5DateUtc'],  # data da corrida
                    'temp_ar_media'   : round(weather['AirTemp'].mean(), 2),
                    'temp_pista_media': round(weather['TrackTemp'].mean(), 2),
                    'umidade_media'   : round(weather['Humidity'].mean(), 2),
                    'corrida_molhada' : int(weather['Rainfall'].any()),
                    'perc_voltas_chuva': round(perc_chuva, 1),
                })

            except Exception as e:
                print(f"Erro {temporada} R{rodada}: {e}")

    return pd.DataFrame(registros)


df_clima = extrair_clima_fastf1(
    temp_inicio  = TEMPORADA_INICIO,
    temp_fim     = TEMPORADA_FIM,
    rodada_inicio= RODADA_INICIO,
    rodada_fim   = RODADA_FIM,
)

output = f"clima_f1_prox_corrida.csv"
df_clima.to_csv(output, index=False, sep=';')
print(f"\nSalvo: {output}")
print(df_clima.head())